### 1. Basic Tasks

In [0]:
%sql
-- 1.
-- define schema
create schema if not exists cyntexa_dev.bronze;
create schema if not exists cyntexa_dev.silver;
create schema if not exists cyntexa_dev.gold;

In [0]:
from pyspark.sql.functions import *

In [0]:
#  Data ingestion to bronze
df = spark.read.csv("/Volumes/cyntexa_dev/sales/raw/sales (1).csv", header = True)
df.write.mode("overwrite").saveAsTable("cyntexa_dev.bronze.sales")

In [0]:
# 2. Clean table and store in silver
df_cleaned = df.withColumn("sale_id", col("sale_id").cast("int")) \
    .withColumn("customer_id", col("customer_id").cast("int")) \
        .withColumn("product_id", col("product_id").cast("int")) \
            .withColumn("quantity", col("quantity").cast("double")) \
                .withColumn("sale_amount", col("sale_amount").cast("double")) \
                    .withColumn("sale_date", to_date(col("sale_date")))\
                        .dropDuplicates().dropna()

df_cleaned.write.mode("overwrite").saveAsTable("cyntexa_dev.silver.sales_cleaned")

In [0]:
%sql
-- 3. Gold tables
create table cyntexa_dev.gold.customer_revenue as
select customer_id, sum(sale_amount) as total_revenue 
from cyntexa_dev.silver.sales_cleaned
group by customer_id
order by sum(sale_amount)


### 2. Intermediate Tasks

In [0]:
# 4.